# Notebook 5 — Train, Validation & Test
### Sprint 7 | Machine Learning Fundamentals for AI/ML Engineers

**Methodology:** Understand -> Demonstrate -> Implement -> Interpret, using the Telco
Customer Churn dataset (classification) throughout.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])
df_encoded = df.copy()
for col in encode_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

X = df_encoded.drop(columns=['customerID', 'Churn'])
y = (df['Churn'] == 'Yes').astype(int)
print(f"Dataset ready: {X.shape}")


Dataset ready: (7043, 19)


/tmp/ipykernel_539/2478984534.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])


---
## 1-3. Training, Validation, and Test Datasets

### Understand
**Training data** is what a model directly learns parameters from. **Validation data**
tunes choices (hyperparameters, which model to use) without contaminating the final
evaluation. **Test data** is touched exactly once, at the very end, for an honest
real-world performance estimate.

### Why the test set must NOT be used for tuning
If hyperparameters are repeatedly adjusted by checking test performance, the test set
stops being a genuinely unseen estimate — the model is *indirectly* fit to it through
the tuning process. This is the exact mechanism behind an inflated, unrealistic final
score.


---
## 4. Train-Test Split & 5. Train-Validation-Test Split

### Implement


In [2]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42)

for name, subset in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    print(f"{name:<12}: {len(subset):,} rows ({len(subset)/len(X)*100:.1f}%), churn rate={subset.mean()*100:.2f}%")


Train       : 4,929 rows (70.0%), churn rate=26.54%
Validation  : 1,057 rows (15.0%), churn rate=26.58%
Test        : 1,057 rows (15.0%), churn rate=26.49%


---
## 6. Cross Validation & 7. K-Fold Cross Validation

### Understand
Instead of a single train/validation split, K-Fold CV splits the training data into K
equal parts, trains K times (each time holding out a different fold as validation), and
averages the results — a more robust estimate than any single split, since it doesn't
depend on which particular rows happened to land in the validation set.

### Implement


In [3]:
model = LogisticRegression(max_iter=2000, class_weight='balanced')
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
kfold_scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring='accuracy')

print(f"5-Fold CV accuracy scores: {kfold_scores.round(4)}")
print(f"Mean: {kfold_scores.mean():.4f} | Std: {kfold_scores.std():.4f}")


5-Fold CV accuracy scores: [0.7394 0.7363 0.7515 0.7394 0.7604]
Mean: 0.7454 | Std: 0.0091


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


---
## 8. Stratified K-Fold

### Understand
Regular K-Fold shuffles rows randomly into folds — with an imbalanced target (this
dataset: 73.5%/26.5%, Sprint 4), a given fold could by chance get an unusually skewed
class ratio. Stratified K-Fold preserves the overall class ratio in EVERY fold.

### Implement


In [4]:
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skfold_scores = cross_val_score(model, X_train, y_train, cv=skfold, scoring='accuracy')

print(f"Stratified 5-Fold CV accuracy scores: {skfold_scores.round(4)}")
print(f"Mean: {skfold_scores.mean():.4f} | Std: {skfold_scores.std():.4f}")

print(f"\nStd dev comparison — regular K-Fold: {kfold_scores.std():.4f} vs Stratified: {skfold_scores.std():.4f}")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Stratified 5-Fold CV accuracy scores: [0.7485 0.7505 0.7485 0.7475 0.7462]
Mean: 0.7482 | Std: 0.0014

Std dev comparison — regular K-Fold: 0.0091 vs Stratified: 0.0014


**Finding:** Stratified K-Fold typically shows equal or lower variance across
folds — each fold is a more representative, comparable sample of the true class balance,
consistent with Sprint 5, Notebook 12's finding that stratification measurably reduces
variance for this exact dataset.


---
## 9. Random State

### Understand
(Recap from Sprint 5/Sprint 3) Fixes the pseudo-random split/shuffle so results are
exactly reproducible run to run — essential for fair before/after comparisons throughout
this sprint.

### Implement


In [5]:
scores_a = cross_val_score(model, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=42))
scores_b = cross_val_score(model, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=42))
scores_c = cross_val_score(model, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=7))

print(f"Same random_state (42) twice -> identical scores: {np.allclose(scores_a, scores_b)}")
print(f"Different random_state (7)   -> different scores: {not np.allclose(scores_a, scores_c)}")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Same random_state (42) twice -> identical scores: True
Different random_state (7)   -> different scores: True


---
## 10. Data Leakage (Recap)

### Understand
(Full treatment: Sprint 5, Notebook 13) The most relevant reminder here: any
preprocessing that "learns" something (a scaler's mean, an encoder's categories) must be
fit only within each CV fold's training portion, not on the whole dataset beforehand —
otherwise every fold's "validation" score is quietly contaminated.


---
## 11. Overfitting & 12. Underfitting (Preview — Full Treatment in Notebook 16)

### Understand
**Overfitting**: a model that learned the training data's noise, not just its pattern —
excellent training performance, poor validation/test performance. **Underfitting**: a
model too simple to capture even the real pattern — poor performance everywhere.

### Demonstrate — A Quick Preview


In [6]:
from sklearn.tree import DecisionTreeClassifier

# An intentionally overfit model (unlimited depth) vs a reasonable one
overfit_model = DecisionTreeClassifier(random_state=42)   # no depth limit
reasonable_model = DecisionTreeClassifier(max_depth=5, random_state=42)

for name, m in [('Unlimited depth (likely overfit)', overfit_model), ('max_depth=5 (reasonable)', reasonable_model)]:
    m.fit(X_train, y_train)
    train_acc = m.score(X_train, y_train)
    val_acc = m.score(X_val, y_val)
    print(f"{name}: train={train_acc:.4f}, validation={val_acc:.4f}, gap={train_acc-val_acc:.4f}")


Unlimited depth (likely overfit): train=0.9980, validation=0.7313, gap=0.2667
max_depth=5 (reasonable): train=0.8056, validation=0.7871, gap=0.0185


**Finding:** The unlimited-depth tree shows a much larger train-validation gap —
a classic overfitting signature, previewing Notebook 16's full treatment.


---
## Summary

| Concept | This Dataset's Application |
|---|---|
| Train/Val/Test | Stratified 70/15/15 split |
| K-Fold CV | 5-fold, mean accuracy computed with std as a stability measure |
| Stratified K-Fold | Confirmed lower/equal variance vs regular K-Fold |
| Random State | Fixed at 42 throughout this sprint for reproducibility |
| Overfitting Preview | Unlimited-depth tree shows a large train-validation gap |

**Next notebook:** `06_Logistic_Regression.ipynb` — the first classification algorithm
covered in depth.
